In [ ]:
import numpy as np
import pandas as pd
from typing import List
import os
import matplotlib.pyplot as plt
from colour import (
    MSDS_CMFS,
    SDS_ILLUMINANTS,
    CCS_ILLUMINANTS,
    SpectralShape,
    SpectralDistribution,
    sd_to_XYZ,
    XYZ_to_xy,
    xy_to_CCT,
)

def read_lss(lss_path):
    """
    Load a numpy array from the given lss file path and convert it to a pandas DataFrame.
    """
    df = pd.read_csv(lss_path, sep="\t", header=0)
    return df


def load_camspecs_specs(file_paths: List[str], suffix: str = "SR"):
    data_list = []
    wave_lengths = None

    assert suffix in ["SR", "QE", "RAW"], suffix

    for file_path in file_paths:
        data = pd.read_csv(file_path, sep='\s+', skiprows=15, encoding="SHIFT-JIS")
        data[['Lambda', f'R_{suffix}', f'G_{suffix}', f'B_{suffix}']] = \
            data[['Lambda', f'R_{suffix}', f'G_{suffix}', f'B_{suffix}']].apply(pd.to_numeric, errors='coerce')

        if wave_lengths is None:
            wave_lengths = data['Lambda'].values
        else:
            assert (data["Lambda"].values == wave_lengths).all

        data_np = data[[f'R_{suffix}', f'G_{suffix}', f'B_{suffix}']].values.T
        data_list.append(data_np)

    data_np = np.array(data_list)
    assert data_np.shape == (len(file_paths), 3, len(wave_lengths)), data_np.shape
    assert wave_lengths.shape == (len(wave_lengths),), wave_lengths.shape

    return data_np, wave_lengths

def load_and_interpolate_munsell_txt(filepath):
    """
    Reads a Munsell-format reflectance text file and interpolates each reflectance block
    from 10nm (380–730nm) to 5nm intervals.

    Parameters:
        filepath (str): Path to the Munsell .txt file.

    Returns:
        reflectances_interp (np.ndarray): Array of interpolated reflectance curves,
                                          shape (num_curves, 71).
        wavelengths_interp (np.ndarray): The interpolated wavelength range (380–730nm in 5nm steps).
    """
    reflectances = []

    with open(filepath, "r") as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line.startswith('"'):  # start of a new reflectance block
            i += 1
            block = []
            for _ in range(36):  # next 36 lines are wavelength-reflectance
                wavelength, value = lines[i].strip().split()
                block.append(float(value))
                i += 1
            reflectances.append(block)
        else:
            i += 1

    reflectances = np.array(reflectances)  # shape: (num_curves, 36)

    wavelengths_orig = np.arange(380, 731, 10)
    wavelengths_interp = np.arange(380, 731, 5)

    reflectances_interp = np.array([
        np.interp(wavelengths_interp, wavelengths_orig, r)
        for r in reflectances
    ])

    return reflectances_interp, wavelengths_interp



def simulate_camera_response(illuminations, reflectances, camera):
    """
    Simulates RGB responses of a camera under various illuminations and surface reflectances.

    Parameters:
        illuminations: np.ndarray of shape (N, W)
            Spectral power distributions for N illuminants over W wavelengths.
        
        reflectances: np.ndarray of shape (M, W)
            Spectral reflectance functions for M surfaces over W wavelengths.
        
        camera: np.ndarray of shape (3, W)
            Camera spectral sensitivity functions (R, G, B) over W wavelengths.

    Returns:
        RGB: np.ndarray of shape (N, M, 3)
            Simulated RGB responses for each (illumination, reflectance) pair.
    """
    # Ensure input shapes are consistent
    assert illuminations.shape[1] == reflectances.shape[1] == camera.shape[1], \
        "All inputs must have the same number of wavelength samples (axis 1)."

    # Compute spectral product: (N, M, W)
    spectra = illuminations[:, np.newaxis, :] * reflectances[np.newaxis, :, :]  # shape: (N, M, W)

    # Multiply with camera: dot along wavelength axis
    RGB = np.tensordot(spectra, camera, axes=([2], [1]))  # result: (N, M, 3)

    return RGB



In [ ]:

illuminations = read_lss("${repo_root}/assets/lss/huang_390 illuminants.lss")
#remove the first column
illuminations = illuminations.iloc[:, 1:]

start_wavelength = 300
end_wavelength = 1000
step = 5

start_idx = int((380 - start_wavelength) / step)  # (380 - 300) / 5 = 16
end_idx = int((730 - start_wavelength) / step)    # (730 - 300) / 5 = 86

illuminations = illuminations.iloc[:, start_idx:end_idx+1]  # +1 because end is exclusive
#convert to numpy array
illuminations = illuminations.to_numpy() #Illuminations * 71
#divide by max illuminations to normalize
illuminations /= np.max(illuminations, axis=1, keepdims=True)

shape_5nm = SpectralShape(380, 730, 5)  # 380 to 730 nm in 5 nm steps
cmfs = MSDS_CMFS["CIE 1931 2 Degree Standard Observer"].copy().align(shape_5nm)
xyz = cmfs.values.T  # shape: (3, 71)
# Normalize so max of any channel is 1
xyz /= np.max(xyz)


reflectances_munsell, wavelengths = load_and_interpolate_munsell_txt("Munsell_1994.txt")
reflectances_xrite, _ = load_and_interpolate_munsell_txt("XriteColorChecker24.txt")

white_reflectances = reflectances_xrite[18]
reflectances_munsell = np.vstack((reflectances_munsell, white_reflectances))


# 2. Load and align D50 illuminant to same spectral shape
d50_sd = SDS_ILLUMINANTS["D50"].copy().align(shape_5nm)
d50_sd = d50_sd.values  # shape: (71,)
#normalize so sum is 1
d50_sd /= np.max(d50_sd)

# Repeat D50 834 times to match the shape of your illuminations
d50_illuminations = np.tile(d50_sd, (834, 1))  # shape: (834, 71)








In [ ]:
from pathlib import Path

camspecs_path = Path("${sim_root}/camspecs_onlyhoang/") 
#get all .txt files in the directory
sensitivity_files = list(camspecs_path.glob("*.txt"))
for sensitivity_file in sensitivity_files:
    camera, wavelengths = load_camspecs_specs([sensitivity_file])
    #wavelengths is 380 to 755nm in 5nm increments

    #clip the data to 380nm to 730nm
    wave_range = (wavelengths >= 380) & (wavelengths <= 730)
    wavelengths = wavelengths[wave_range]
    camera = camera[0, :, wave_range]
    camera = camera.T #3x71
    camera /= np.max(camera)

    print("max of camera:", np.max(camera, axis=1))

   
    camera_RGB = simulate_camera_response(illuminations, reflectances_munsell, camera)
    np.save(camspecs_path / f"{sensitivity_file.stem}_simulated.npy", camera_RGB)

#max of xyz reflectances
print("Max of xyz reflectances:", np.max(xyz))

print("Max of munsell reflectances:", np.max(reflectances_munsell))

print("Max of d50 illuminations:", np.max(d50_illuminations))

print("max of illuminations:", np.max(illuminations))

XYZ =  simulate_camera_response(d50_illuminations, reflectances_munsell, xyz)  #XYZ - this happens 


#save to camspecs_path/xyz_simulated.npy
np.save(camspecs_path / "xyz_simulated.npy", XYZ)


In [ ]:
# #load in sonyold_simulated.npy andcamspecs/sony_simulated.npy and compare
# sony_old = np.load("old_simulated.npy")
# sony_new = np.load(camspecs_path / "sony_simulated.npy")
# print("Sony old shape:", sony_old.shape)
# print("Sony new shape:", sony_new.shape)
# print("Difference between old and new Sony simulations:", np.abs(sony_old - sony_new).max())